# pandas

## 基本

In [23]:
# テスト用データ生成
import pandas as pd
import copy

data = [['A', 100], ['B', 30], ['C', 150]]
_df = pd.DataFrame(data, columns=['name', 'value'])
_df

,name,value
0,A,100
1,B,30
2,C,150


TODO
* column操作系
* index操作系
* concat系
* merge系
* loc, iloc 値の取り出し　値の上書き
* 列単位への関数適応



In [4]:
# 特定の列をインデックスに指定する
df = copy.deepcopy(_df)
df.set_index('name')

,value
name,
A,100
B,30
C,150


行、列を追加する場合、以下の3手段をとることができる。
* 値を一括で追加（単一）
* 値をリストで指定して追加（リスト）
* 不足分はNanで代入する（series）

concatも使用できるが、今回は除外する。
<https://takilog.com/pandas-dataframe-append-concat/>

In [ ]:
# 列の追加 値は一括で入力
df = copy.deepcopy(_df)
df['test'] = 0
df

,name,value,test
0,A,100,0
1,B,30,0
2,C,150,0


In [10]:
# 列の追加 値はlistで入力
df = copy.deepcopy(_df)
df['test'] = [1, 2, 3]
df

,name,value,test
0,A,100,1
1,B,30,2
2,C,150,3


In [12]:
# 列の追加 値が足りない場合はNanで追加
df = copy.deepcopy(_df)
data = [1, 3]
sr = pd.Series(data, index=[0, 2])
df['test'] = sr
df

,name,value,test
0,A,100,1.0
1,B,30,NaN
2,C,150,3.0


In [ ]:
# 行を追加する　値は一括
df = copy.deepcopy(_df)
df.loc['s'] = 0
# df.loc['s', :] = 0
df

,name,value
0,A,100
1,B,30
2,C,150
s,0,0


In [19]:
# 行を追加する　値はlistで指定
df = copy.deepcopy(_df)
df.loc['s'] = ['d', 34]
# df.loc['s', :] = ['d', 34]
df

,name,value
0,A,100
1,B,30
2,C,150
s,d,34


In [ ]:
# 行を追加する　series
df = copy.deepcopy(_df)
data = ['f', 32]
sr = pd.Series(data, index=['name', 'value'])

data = ['f']
sr = pd.Series(data, index=['name'])

df.loc[3] = sr
df

,name,value
0,A,100.0
1,B,30.0
2,C,150.0
3,f,NaN


## [tips] iterrowsと処理速度

<https://qiita.com/141sksk/items/9883be05a3851c90d1d1>

テスト用データ準備

In [1]:
from sklearn.datasets import load_iris
iris = load_iris()

In [2]:
iris.data.shape

(150, 4)

In [3]:
import pandas as pd
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)

In [4]:
df_iris.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


iterrowsは便利で分かりやすいが、遅い、、

In [9]:
%%timeit
for idx, row in df_iris.iterrows():
    if row[0] < 4:
        row[0] = 1


4.86 ms ± 410 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


listに変換すると、かなり速い

In [18]:
list_iris = df_iris.values.tolist()
list_iris[0:3]

[[5.1, 3.5, 1.4, 0.2], [4.9, 3.0, 1.4, 0.2], [4.7, 3.2, 1.3, 0.2]]

In [17]:
%%timeit
for ele in list_iris:
    if ele[0] < 4:
        ele[0] = 1

8.61 µs ± 121 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


dictでもだいぶ速い

In [23]:
dict_iris = df_iris.to_dict()
len(dict_iris.values())
dict_iris.keys()


dict_keys(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)'])

In [24]:
%%timeit
for key, val in df_iris.to_dict().items():
    if key == 'sepal length (cm)':
        val[0] = 3

311 µs ± 20.3 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


applyもdictくらい速い

In [25]:
def map_oz(col):
    if col < 4:
        return 1
    return 0

In [27]:
%%timeit
df_iris['petal length (cm)'].apply(map_oz)

248 µs ± 21.6 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
